In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from collections import Counter
from skimage.color import rgb2gray
from skimage.transform import resize
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import RandomOverSampler
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import optuna
from optuna.visualization.matplotlib import plot_param_importances
import torch.optim as optim
import torch.nn as nn
from torchvision import models
import torchvision.models as models
import numpy as np

In [ ]:
# Base directories 
base_dir = 'chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

classes = ['NORMAL', 'PNEUMONIA']

In [ ]:
def explore_data(directory, classes):
    data_count = {}
    for cls in classes:
        path = os.path.join(directory, cls)
        jpeg_files = [file for file in os.listdir(path) if file.lower().endswith(('.jpeg', '.jpg'))]
        data_count[cls] = len(jpeg_files)
    return data_count

train_distribution = explore_data(train_dir, classes)
val_distribution = explore_data(val_dir, classes)
test_distribution = explore_data(test_dir, classes)

print("Training set:", train_distribution)
print("Validation set:", val_distribution)
print("Test set:", test_distribution)


In [ ]:
def plot_distribution(data, title):
    plt.bar(data.keys(), data.values(), color=['skyblue', 'salmon'])
    plt.title(title)
    plt.ylabel('Number of images')
    plt.xlabel('Classes')
    plt.show()

plot_distribution(train_distribution, 'Training Set Distribution')
plot_distribution(val_distribution, 'Validation Set Distribution')
plot_distribution(test_distribution, 'Test Set Distribution')


In [ ]:
def check_class_balance(data_count):
    total = sum(data_count.values())
    balance = {cls: (count / total) * 100 for cls, count in data_count.items()}
    return balance

print("Training set class balance (%):", check_class_balance(train_distribution))
print("Validation set class balance (%):", check_class_balance(val_distribution))
print("Test set class balance (%):", check_class_balance(test_distribution))


In [ ]:
def load_images_labels(directory, classes, img_size=(224, 224)):
    images, labels = [], []
    valid_extensions = ('.jpeg', '.jpg')
    for label, cls in enumerate(classes):
        cls_path = os.path.join(directory, cls)
        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith(valid_extensions):
                img_path = os.path.join(cls_path, img_name)
                try:
                    img = Image.open(img_path).convert('RGB')
                    img = img.resize(img_size)
                    images.append(np.array(img))
                    labels.append(label)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")
    return np.array(images), np.array(labels)

X_train, y_train = load_images_labels(train_dir, classes)
X_val, y_val = load_images_labels(val_dir, classes)
X_test, y_test = load_images_labels(test_dir, classes)

print(f"Training set size: {X_train.shape}, Labels: {Counter(y_train)}")
print(f"Validation set size: {X_val.shape}, Labels: {Counter(y_val)}")
print(f"Test set size: {X_test.shape}, Labels: {Counter(y_test)}")


In [ ]:
def show_sample_images(directory, classes, samples_per_class=3):
    valid_extensions = ('.jpeg', '.jpg')
    fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(samples_per_class*4, len(classes)*4))
    
    for i, cls in enumerate(classes):
        cls_path = os.path.join(directory, cls)
        image_files = [img for img in os.listdir(cls_path) if img.lower().endswith(valid_extensions)]
        
        for j, img_name in enumerate(image_files[:samples_per_class]):
            img_path = os.path.join(cls_path, img_name)
            try:
                img = Image.open(img_path)
                axes[i, j].imshow(img, cmap='gray')
                axes[i, j].axis('off')
                axes[i, j].set_title(f'{cls}')
            except Exception as e:
                print(f"Error loading image {img_path}: {e}")
    plt.tight_layout()
    plt.show()

show_sample_images(train_dir, classes)


In [ ]:
def resolution_statistics(directory, classes):
    resolutions = []
    valid_extensions = ('.jpeg', '.jpg')
    
    for cls in classes:
        cls_path = os.path.join(directory, cls)
        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith(valid_extensions):
                img_path = os.path.join(cls_path, img_name)
                try:
                    with Image.open(img_path) as img:
                        resolutions.append(img.size)
                except Exception as e:
                    print(f"Skipped {img_path} due to error: {e}")
    
    if resolutions:
        resolutions = np.array(resolutions)
        widths, heights = resolutions[:, 0], resolutions[:, 1]
        print(f"Min resolution: {min(widths)}x{min(heights)}")
        print(f"Max resolution: {max(widths)}x{max(heights)}")
        print(f"Average resolution: {np.mean(widths):.1f}x{np.mean(heights):.1f}")
    else:
        print("No valid images found.")


resolution_statistics(train_dir, classes)


In [ ]:
def check_corrupted_images(directory, classes):
    corrupted_images = []
    for cls in classes:
        cls_path = os.path.join(directory, cls)
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except (IOError, SyntaxError) as e:
                corrupted_images.append(img_path)
    if corrupted_images:
        print("Corrupted images detected:", corrupted_images)
    else:
        print("No corrupted images detected.")

check_corrupted_images(train_dir, classes)


In [ ]:
def pixel_intensity_summary(directory, classes, num_images_per_class=50):
    pixel_means = []
    pixel_stds = []
    valid_extensions = ('.jpeg', '.jpg')
    
    for cls in classes:
        cls_path = os.path.join(directory, cls)
        images = [img for img in os.listdir(cls_path) if img.lower().endswith(valid_extensions)]
        for img_name in images[:num_images_per_class]:
            img_path = os.path.join(cls_path, img_name)
            try:
                img = Image.open(img_path).convert('L')  # grayscale
                pixels = np.array(img).flatten()
                pixel_means.append(np.mean(pixels))
                pixel_stds.append(np.std(pixels))
            except Exception as e:
                print(f"Skipped {img_path} due to error: {e}")
    
    if pixel_means and pixel_stds:
        print(f"Mean pixel intensity (sample): {np.mean(pixel_means):.2f}")
        print(f"Std deviation pixel intensity (sample): {np.mean(pixel_stds):.2f}")
    else:
        print("No valid images were found to compute pixel intensity.")


pixel_intensity_summary(train_dir, classes)


In [ ]:
def preprocess_classical(X, img_size=(128, 128)):
    # Resize and grayscale
    processed = []
    for img in X:
        img_pil = Image.fromarray(img).convert('L').resize(img_size)  # Grayscale
        processed.append(np.array(img_pil).flatten())  # Flatten
    processed = np.array(processed)
    
    # Scale features
    scaler = StandardScaler()
    processed = scaler.fit_transform(processed)
    
    return processed, scaler


In [ ]:
def preprocess_with_hog(X, img_size=(128, 128), pixels_per_cell=(8, 8), cells_per_block=(2, 2), orientations=9):
    hog_features = []

    for img in X:
        img_resized = resize(rgb2gray(img), img_size, anti_aliasing=True)
        
        # Extract HOG features
        features = hog(
            img_resized,
            orientations=orientations,
            pixels_per_cell=pixels_per_cell,
            cells_per_block=cells_per_block,
            block_norm='L2-Hys',
            visualize=False
        )
        hog_features.append(features)
    
    hog_features = np.array(hog_features)

    # Scale features
    scaler = StandardScaler()
    hog_features_scaled = scaler.fit_transform(hog_features)
    
    return hog_features_scaled, scaler


In [ ]:
# Preprocess with HOG
X_train_hog, scaler = preprocess_with_hog(X_train)

# Apply same preprocessing to val/test
X_val_hog = scaler.transform([
    hog(resize(rgb2gray(x), (128, 128)), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys') 
    for x in X_val
])

X_test_hog = scaler.transform([
    hog(resize(rgb2gray(x), (128, 128)), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys') 
    for x in X_test
])

print(f"Train shape: {X_train_hog.shape}")
print(f"Validation shape: {X_val_hog.shape}")
print(f"Test shape: {X_test_hog.shape}")


In [ ]:
# selection of the 3000 best feautures
selector = SelectKBest(score_func=f_classif, k=3000)
X_train_kbest = selector.fit_transform(X_train_hog, y_train)
X_val_kbest = selector.transform(X_val_hog)
X_test_kbest = selector.transform(X_test_hog)

print("Reduced shapes:")
print("Train:", X_train_kbest.shape)
print("Val:", X_val_kbest.shape)
print("Test:", X_test_kbest.shape)


In [ ]:
# Apply oversampling to balance the HOG-transformed training data
ros = RandomOverSampler(random_state=42)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_kbest, y_train)

print(f"Original training set shape: {Counter(y_train)}")
print(f"Balanced training set shape: {Counter(y_train_balanced)}")


In [ ]:
# Mean and std can be adjusted based on your own dataset later
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset   = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(test_dir, transform=val_test_transforms)


In [ ]:
# Temp loader with out normalization
temp_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])
temp_dataset = datasets.ImageFolder(train_dir, transform=temp_transform)
temp_loader = DataLoader(temp_dataset, batch_size=64, shuffle=False)

# mean & std
mean = 0.
std = 0.
n_images = 0

for images, _ in temp_loader:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    n_images += batch_samples

mean /= n_images
std /= n_images

print("Dataset mean:", mean)
print("Dataset std:", std)


In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


In [ ]:
# Check a sample batch from the training loader

def show_batch(dataloader, classes):
    images, labels = next(iter(dataloader))
    fig, ax = plt.subplots(1, 6, figsize=(15, 4))
    for i in range(6):
        img = images[i].permute(1, 2, 0) * torch.tensor(imagenet_std) + torch.tensor(imagenet_mean)  # de-normalize
        img = torch.clamp(img, 0, 1)
        ax[i].imshow(img)
        ax[i].set_title(classes[labels[i]])
        ax[i].axis('off')
    plt.show()

show_batch(train_loader, classes)


In [ ]:
# Train Random Forest using balanced data
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_balanced, y_train_balanced)  # Use balanced HOG data

# Predict on HOG-transformed test set
rf_preds = rf_model.predict(X_test_kbest)

# Evaluation
print("Random Forest Accuracy (Balanced, HOG features):", accuracy_score(y_test, rf_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_preds))
print("\nClassification Report:\n", classification_report(y_test, rf_preds))


In [ ]:
# Train basic SVM on balanced HOG features
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(X_train_balanced, y_train_balanced)  # Use balanced HOG features

# Predict on HOG-transformed test set
svm_preds = svm_model.predict(X_test_kbest)

# Evaluation
print("SVM Accuracy (Balanced, HOG features):", accuracy_score(y_test, svm_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, svm_preds))
print("\nClassification Report:\n", classification_report(y_test, svm_preds))


In [ ]:
def plot_confusion(cm, title):
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

# Plot for Random Forest
plot_confusion(confusion_matrix(y_test, rf_preds), "Random Forest Confusion Matrix")

# Plot for SVM
plot_confusion(confusion_matrix(y_test, svm_preds), "SVM Confusion Matrix")


In [ ]:
#  Objective Function
def rf_objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 5, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train_balanced, y_train_balanced)
    preds = clf.predict(X_val_kbest)
    return accuracy_score(y_val, preds)

#  Run Optuna
rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)


#  Plot Hyperparameter Importance
fig = plot_param_importances(rf_study)
plt.tight_layout()
plt.show()

#  Retrain Best RF on Full Train Data
best_params = rf_study.best_params
print("Best Parameters from Optuna:", best_params)

best_rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
best_rf.fit(X_train_balanced, y_train_balanced)

#  Final Evaluation on Test Set
rf_optuna_preds = best_rf.predict(X_test_kbest)
print(" Accuracy:", accuracy_score(y_test, rf_optuna_preds))
print("\n Confusion Matrix:\n", confusion_matrix(y_test, rf_optuna_preds))
print("\n Classification Report:\n", classification_report(y_test, rf_optuna_preds))


In [ ]:
def run_rf_with_params(n_estimators, max_depth, min_samples_split, min_samples_leaf, class_weight):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight=class_weight,
        random_state=42
    )
    
    model.fit(X_train_balanced, y_train_balanced)
    preds = model.predict(X_val_kbest)
    return accuracy_score(y_val, preds)


In [ ]:
def refined_objective(trial):
    return run_rf_with_params(
        n_estimators = trial.suggest_int("n_estimators", 30, 200),
        max_depth = trial.suggest_int("max_depth", 10, 40),
        min_samples_split = 4,  
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 3, 10),
        class_weight = "balanced" 
    )

refined_study = optuna.create_study(direction="maximize")
refined_study.optimize(refined_objective, n_trials=30)



print(" Refined Best Parameters:", refined_study.best_params)


In [ ]:
# Unpack refined best params
best_params = refined_study.best_params


final_rf = RandomForestClassifier(
    n_estimators=252,  
    max_depth=best_params['max_depth'],
    min_samples_split=4,
    min_samples_leaf=7,  
    class_weight="balanced",  
    random_state=42
)

# Train and evaluate
final_rf.fit(X_train_balanced, y_train_balanced)
final_preds = final_rf.predict(X_test_kbest)

# Final evaluation
print(" Final RF Accuracy on Test Set:", accuracy_score(y_test, final_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, final_preds))
print("\nClassification Report:\n", classification_report(y_test, final_preds))


In [ ]:
class ConfigurableCNN(nn.Module):
    def __init__(self,
                 num_classes,
                 activation="relu",
                 use_batch_norm=True,
                 dropout_rate=0.5,
                 num_conv_layers=2,
                 conv_channels=[32, 64],
                 num_linear_layers=1,
                 linear_units=[512]):
        super(ConfigurableCNN, self).__init__()

        self.activation_func = self.get_activation_function(activation)
        self.use_batch_norm = use_batch_norm
        self.dropout = nn.Dropout(dropout_rate)

        self.conv_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList() if self.use_batch_norm else None

        in_channels = 3
        for i in range(num_conv_layers):
            out_channels = conv_channels[i]
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=1)
            self.conv_layers.append(conv)
            if self.use_batch_norm:
                self.bn_layers.append(nn.BatchNorm2d(out_channels))
            in_channels = out_channels

        self.pool = nn.AdaptiveAvgPool2d((7, 7))

        self.fc_layers = nn.ModuleList()
        in_features = conv_channels[-1] * 7 * 7
        for i in range(num_linear_layers):
            out_features = linear_units[i]
            fc = nn.Linear(in_features, out_features)
            self.fc_layers.append(fc)
            in_features = out_features

        self.output_layer = nn.Linear(in_features, num_classes)

    def get_activation_function(self, activation):
        activations = {
            "relu": nn.ReLU(),
            "leaky_relu": nn.LeakyReLU(0.01),
            "elu": nn.ELU(),
            "swish": nn.SiLU()
        }
        return activations.get(activation, nn.ReLU())

    def forward(self, x):
        for i, conv in enumerate(self.conv_layers):
            x = conv(x)
            if self.use_batch_norm:
                x = self.bn_layers[i](x)
            x = self.activation_func(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        for fc in self.fc_layers:
            x = fc(x)
            x = self.activation_func(x)
            x = self.dropout(x)
        return self.output_layer(x)


In [ ]:
def get_optimizer(optimizer_name, model, learning_rate=1e-3, weight_decay=1e-4):
    optimizers = {
        "SGD": optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay),
        "RMSprop": optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
        "Adam": optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
        "AdamW": optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
        "Adagrad": optim.Adagrad(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
    }
    return optimizers.get(optimizer_name, optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay))

class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return torch.mean(focal_loss)
        elif self.reduction == 'sum':
            return torch.sum(focal_loss)
        return focal_loss


In [ ]:
def train_model_optuna(trial):
    activation = trial.suggest_categorical("activation", ["relu", "leaky_relu", "elu", "swish"])
    optimizer_name = trial.suggest_categorical("optimizer_name", ["SGD", "RMSprop", "Adam", "AdamW", "Adagrad"])
    use_batch_norm = trial.suggest_categorical("use_batch_norm", [True, False])
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 5)
    conv_channels = [trial.suggest_categorical(f"conv_channels_{i}", [16, 32, 64, 128]) for i in range(num_conv_layers)]

    num_linear_layers = trial.suggest_int("num_linear_layers", 1, 2)
    linear_units = [trial.suggest_categorical(f"linear_units_{i}", [256, 512, 1024]) for i in range(num_linear_layers)]

    loss_function = trial.suggest_categorical("loss_function", ["cross_entropy", "focal_loss"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = 2

    model = ConfigurableCNN(
        num_classes=num_classes,
        activation=activation,
        use_batch_norm=use_batch_norm,
        dropout_rate=dropout_rate,
        num_conv_layers=num_conv_layers,
        conv_channels=conv_channels,
        num_linear_layers=num_linear_layers,
        linear_units=linear_units
    ).to(device)

    optimizer = get_optimizer(optimizer_name, model, learning_rate, weight_decay)
    criterion = FocalLoss() if loss_function == "focal_loss" else nn.CrossEntropyLoss()

    patience = 3
    best_val_loss = float("inf")
    best_val_acc = 0.0
    no_improvement_epochs = 0
    best_model_state = None

    for epoch in range(30):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100.0 * correct / total

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100.0 * correct / total

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc  
            best_model_state = model.state_dict()
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1
            if no_improvement_epochs >= patience:
                break

        print(f"Epoch {epoch+1}: Train Acc = {train_acc:.2f}%, Val Acc = {val_acc:.2f}%")

    trial.set_user_attr("best_model_state", best_model_state)
    return best_val_acc  


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(train_model_optuna, n_trials=30)

print("\n Best Hyperparameters Found:")
print(study.best_params)
print(f"Best Validation Accuracy: {study.best_value:.2f}%")


In [ ]:
# Load best hyperparameters from Optuna
best_params = study.best_params

# Reconstruct best CNN model
best_model = ConfigurableCNN(
    num_classes=2,
    activation=best_params["activation"],
    use_batch_norm=best_params["use_batch_norm"],
    dropout_rate=best_params["dropout_rate"],
    num_conv_layers=best_params["num_conv_layers"],
    conv_channels=[best_params[f"conv_channels_{i}"] for i in range(best_params["num_conv_layers"])],
    num_linear_layers=best_params["num_linear_layers"],
    linear_units=[best_params[f"linear_units_{i}"] for i in range(best_params["num_linear_layers"])]
).to("cuda" if torch.cuda.is_available() else "cpu")

# Load the best weights
best_model.load_state_dict(study.best_trial.user_attrs["best_model_state"])
best_model.eval()

# Determine device properly
device = next(best_model.parameters()).device

# Evaluate on the test set
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = best_model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print evaluation results
print(" Final CNN Accuracy on Test Set:", accuracy_score(all_labels, all_preds))
print("\n Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))
print("\n Classification Report:\n", classification_report(all_labels, all_preds))


In [ ]:
# Trial 14 Best Observed Config 
manual_best_params = {
    'activation': 'elu',
    'optimizer_name': 'AdamW',
    'use_batch_norm': True,
    'dropout_rate': 0.4153944081128973,
    'weight_decay': 3.993446571005504e-05,
    'learning_rate': 0.0020342860188173,
    'num_conv_layers': 2,
    'conv_channels_0': 16,
    'conv_channels_1': 128,
    'num_linear_layers': 1,
    'linear_units_0': 512,
    'loss_function': 'cross_entropy'
}

# Build the model 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_manual = ConfigurableCNN(
    num_classes=2,
    activation=manual_best_params["activation"],
    use_batch_norm=manual_best_params["use_batch_norm"],
    dropout_rate=manual_best_params["dropout_rate"],
    num_conv_layers=manual_best_params["num_conv_layers"],
    conv_channels=[manual_best_params["conv_channels_0"], manual_best_params["conv_channels_1"]],
    num_linear_layers=manual_best_params["num_linear_layers"],
    linear_units=[manual_best_params["linear_units_0"]]
).to(device)

# Optimizer and loss 
optimizer = get_optimizer(manual_best_params["optimizer_name"], model_manual,
                          manual_best_params["learning_rate"], manual_best_params["weight_decay"])

criterion = nn.CrossEntropyLoss()

# Retrain model 
epochs = 8  
for epoch in range(epochs):
    model_manual.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_manual(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

model_manual.eval()

# Evaluate on test set 
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_manual(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Final Report 
print("\n Evaluation for Manual Trial 14 (Val Acc reached 93.75% during training)")
print(" Accuracy on Test Set:", accuracy_score(all_labels, all_preds))
print("\n Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))
print("\n Classification Report:\n", classification_report(all_labels, all_preds))


In [ ]:
# Define class names if needed
classes = ['NORMAL', 'PNEUMONIA']

def plot_confusion(cm, title):
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

# Compute and plot
cm = confusion_matrix(all_labels, all_preds)
plot_confusion(cm, "CNN Confusion Matrix (Manual Trial 14)")


In [ ]:
def get_feature_extractor_resnet(num_classes=2):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False  # Freeze all layers
    model.fc = nn.Linear(model.fc.in_features, num_classes)  
    return model


In [ ]:
def train_model(model, train_loader, val_loader, name="Model", epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_acc = correct / total * 100

        # Validation
        model.eval()
        correct_val, total_val, val_loss = 0, 0, 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (preds == labels).sum().item()

        val_acc = correct_val / total_val * 100
        val_loss /= len(val_loader)

        is_better = (val_acc > best_val_acc) or (val_acc == best_val_acc and val_loss < best_val_loss)
        if is_better:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_model_state = model.state_dict()

        print(f"[{name}] Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, best_val_acc


In [ ]:
model_frozen = get_feature_extractor_resnet(num_classes=2)
model_frozen, best_val_acc_frz = train_model(
    model_frozen, train_loader, val_loader, name="Feature Extractor ResNet18", epochs=30
)

print("\nBest Validation Accuracy:")
print(f"Feature Extractor ResNet18: {best_val_acc_frz:.2f}%")


In [ ]:
def evaluate_model(model, test_loader, model_name="Model"):
    model.eval()
    device = next(model.parameters()).device
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nEvaluation - {model_name}")
    print(f"Accuracy: {acc * 100:.2f}%")
    print("Classification Report:\n", classification_report(all_labels, all_preds, target_names=["NORMAL", "PNEUMONIA"]))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["NORMAL", "PNEUMONIA"],
                yticklabels=["NORMAL", "PNEUMONIA"])
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()


In [ ]:
evaluate_model(model_frozen, test_loader, model_name="Feature Extractor ResNet18")
